In [ ]:
import sys, os
import pandas as pd

sys.path.append("..")

In [ ]:
from src.config import ESTACIONES, CONTAMINANTES
from src.sinca import actualizar_estaciones, traer_estaciones

In [ ]:
ARCHIVO = "../data/crudo.parquet"
os.makedirs("../data", exist_ok=True)

# 1) Estaciones del config que aún no están en el crudo: se baja su historial completo
if os.path.exists(ARCHIVO):
    existente = pd.read_parquet(ARCHIVO)
    nuevas = {c: v for c, v in ESTACIONES.items() if c not in set(existente["codigo"])}
    if nuevas:
        print("Estaciones nuevas:", [nombre for nombre, _ in nuevas.values()])
        hoy = pd.Timestamp.today().strftime("%y%m%d")
        historial = traer_estaciones(nuevas, CONTAMINANTES, "110101", hoy)
        if historial is not None:
            pd.concat([existente, historial], ignore_index=True).to_parquet(ARCHIVO, index=False)

# 2) Actualización normal de todas las estaciones
datos = actualizar_estaciones(ESTACIONES, CONTAMINANTES, ARCHIVO)

In [ ]:
print("Estaciones:", datos["estacion"].nunique())
print("Parámetros:", sorted(datos["parametro"].unique()))
print("Rango:", datos["timestamp"].min(), "→", datos["timestamp"].max())